# A5 — Tier 2 trên toàn bộ 307 cửa sổ (Phước)

Mục tiêu: **A5a = dry run + pilot 10 cửa sổ** (đo VRAM, s/cửa sổ, chứng minh resume), rồi **A5b = full 307 cửa sổ**
→ `results/verify/<video_id>/window_NNNN.json`. Đây là hạng mục **bắt buộc của đề tài** (tier 2 = VLM sinh ngôn ngữ thật).

Điều kiện trên `main`: A3 (#27) + A3b (#30: chunk ≤ 30 s, `elapsed_s`, `--shard`) — cell 2 **dừng ngay** nếu thiếu.
Fix #35 (cắt cửa sổ trôi quá cuối video) chỉ cần cho **A5b (full 307)**: A5a (dry run + pilot `--limit 10`) chạy được khi chưa có #35,
cell 2 chỉ cảnh báo; cell full run (cell 10) **dừng ngay** nếu thiếu — thiếu #35 thì `RoadAccidents002` và `Vandalism017` sẽ lỗi.

Lượt chạy thật 31/10 (T4, Qwen2.5-VL-3B-Instruct 4-bit, k=4): **307/307 cửa sổ, 114 video, 0 lỗi**, 129 cửa sổ
được VLM xác nhận (42.0%); kết quả ở PR #37 (`results/verify`, 0.56 MB). Notebook này để tái lập từ đầu.

Runtime: **T4 GPU** (Runtime > Change runtime type). Kết quả ghi thẳng vào **Drive** để mất session không mất
tiến độ (runner resume theo fingerprint).


In [ ]:
# Cell 1 — GPU. Notebook đích: T4; L4/A100 chạy được nhưng ngưỡng thời gian bên dưới chỉ hiệu lực trên T4.
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Cần GPU: Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
IS_T4 = name.startswith("Tesla T4")
print(f"GPU: {name} | VRAM {vram_gb:.1f} GB | IS_T4={IS_T4}")
assert vram_gb >= 14, f"Cần >= 14 GB VRAM cho 4-bit 3B + 4 frame/cửa sổ, thấy {name} {vram_gb:.1f} GB"
if not IS_T4:
    print("CẢNH BÁO: không phải T4 — median/p90 s/cửa sổ đo trên T4, không so trực tiếp với A1.")


In [ ]:
# Cell 2 — clone main (đã merge A3 #27 + A3b #30) và kiểm tra A3/A3b (bắt buộc) và #35 (chỉ cảnh báo, cần cho A5b).
import os
import subprocess
import sys

REPO = "/content/vlm-anomalies-detection"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/Kyosaki6/vlm-anomalies-detection.git", REPO], check=True)
os.chdir(REPO)
subprocess.run(["git", "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True, check=True).stdout)

assert os.path.isfile("scripts/run_verify.py"), "Thiếu scripts/run_verify.py — main chưa merge A3 (#27)"
src = open("scripts/run_verify.py", encoding="utf-8").read()
assert "--shard" in src, "main chưa merge A3b (#30: --shard) — pull lại trước khi chạy"
if "clamp_window_to_video" not in src:
    print("CẢNH BÁO: main chưa merge #35 (cắt cửa sổ trôi quá cuối video). A5a (pilot 10 cửa sổ) vẫn chạy được; "
          "A5b full 307 sẽ lỗi ở RoadAccidents002 + Vandalism017 — cell full run sẽ dừng nếu vẫn thiếu #35.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-vlm.txt"], check=True)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
print("\n".join(line for line in freeze.splitlines()
                if line.startswith(("torch==", "transformers==", "bitsandbytes==", "accelerate=="))))


In [ ]:
# Cell 3 — weight VLM (~6.5 GB, KHÔNG tính vào thời gian A5).
from huggingface_hub import snapshot_download

print("weights:", snapshot_download("Qwen/Qwen2.5-VL-3B-Instruct"))


In [ ]:
# Cell 4 — 114 video CÓ cửa sổ; xem dung lượng trước khi tải.
import json

windows = json.load(open("results/windows/_index.json", encoding="utf-8-sig"))["windows"]
print(f"{len(windows)} cửa sổ / {len({w['video_id'] for w in windows})} video cần có")
!df -h /content | tail -1
# Downloader không có --only nên tải cả 290 video test (~8.7 GB, gồm cả calib); A5 chỉ dùng 114 video này.
!python scripts/download_ucf_crime.py --dry-run | tail -3


In [ ]:
# Cell 5 — TẢI VIDEO (resumable: chạy lại = tiếp tục).
!python scripts/download_ucf_crime.py --threads 8

import csv
import json
import os

need_set = {w["video_id"] for w in json.load(open("results/windows/_index.json", encoding="utf-8-sig"))["windows"]}
rows = list(csv.DictReader(open("manifests/ucf_test.csv", encoding="utf-8-sig")))
missing = [r["video_id"] for r in rows if r["video_id"] in need_set and not os.path.exists(r["path"])]
print("thiếu video có cửa sổ:", len(missing), missing[:10])
assert not missing, f"Thiếu {len(missing)} video — không chạy A5b khi thiếu (fail loud, không skip)"


In [ ]:
# Cell 6 — Drive: out-dir TRÊN DRIVE để mất session không mất kết quả.
import os
from google.colab import drive

drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/vlm_a5"
OUT = os.path.join(ROOT, "verify")
LOG = os.path.join(ROOT, "run_a5b.log")
os.makedirs(OUT, exist_ok=True)
print("OUT =", OUT)
print("LOG =", LOG)


In [ ]:
# Cell 7 — DRY RUN: đếm cửa sổ, KHÔNG load model, không ghi gì. Kỳ vọng: đúng 307 cửa sổ.
import subprocess
import sys

dry = subprocess.run(
    [sys.executable, "scripts/run_verify.py",
     "--manifest", "manifests/ucf_test.csv",
     "--windows-dir", "results/windows",
     "--out-dir", OUT, "--k", "4", "--dry-run"],
    capture_output=True, text=True,
)
print("\n".join(dry.stdout.splitlines()[-5:]))
assert dry.returncode == 0, f"dry-run rc={dry.returncode}\n{dry.stderr[-1000:]}"
assert "xong: 307 window xét" in dry.stdout, "kỳ vọng 307 cửa sổ — kiểm tra results/windows/_index.json"


## A5a — pilot 10 cửa sổ (thật)

Chạy **in-process** (`main(argv)`) để đo được `torch.cuda.max_memory_allocated()`; thời gian mỗi cửa sổ lấy từ
`metadata["elapsed_s"]` do runner ghi (A3b), **không** dùng mtime file — mtime sai khi session chết rồi resume.


In [ ]:
# Cell 8 — A5a: verify thật 10 cửa sổ + đo VRAM/s-cửa sổ.
import json
import math
import statistics
import sys
import time
from pathlib import Path

sys.path.insert(0, ".")
import torch
from scripts.run_verify import main

argv = ["--manifest", "manifests/ucf_test.csv", "--windows-dir", "results/windows",
        "--out-dir", OUT, "--k", "4", "--continue-on-error",
        "--verifier-model", "Qwen/Qwen2.5-VL-3B-Instruct", "--limit", "10"]

before = {p: p.stat().st_mtime_ns for p in Path(OUT).rglob("window_*.json")}
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
rc = main(argv)
wall = time.perf_counter() - t0

files = sorted(Path(OUT).rglob("window_*.json"))
new = [p for p in files if p not in before]
elapsed = sorted(json.loads(p.read_text(encoding="utf-8-sig"))["metadata"]["elapsed_s"] for p in new)
p90 = elapsed[min(len(elapsed) - 1, math.ceil(0.9 * len(elapsed)) - 1)] if elapsed else None
pilot = {
    "rc": rc,
    "n_files": len(files),
    "new_this_run": len(new),
    "wall_s": round(wall, 1),
    "median_s_per_window": round(statistics.median(elapsed), 2) if elapsed else None,
    "p90_s_per_window": round(p90, 2) if p90 is not None else None,
    "peak_alloc_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    "peak_reserved_gb": round(torch.cuda.max_memory_reserved() / 2**30, 2),
}
print("PILOT", json.dumps(pilot, ensure_ascii=False))
assert rc in (0, 2), f"run_verify lỗi: {rc}"
assert len(files) >= 10, f"mới {len(files)} file, kỳ vọng 10"
if elapsed and IS_T4:
    # A1 đo 4.45 s/cửa sổ trên T4; trong khoảng 2x là OK. Ngưỡng này chỉ hiệu lực trên T4.
    assert pilot["median_s_per_window"] < 9, "chậm bất thường so với A1 (4.45 s/cửa sổ trên T4)"
if new:
    assert pilot["peak_alloc_gb"] < 5.5, "peak VRAM vượt ngưỡng A1 (5.5 GB)"


In [ ]:
# Cell 9 — CHỨNG MINH RESUME: chạy lại y hệt -> không verify lại cửa sổ nào.
from pathlib import Path


def snapshot():
    return {p: (p.stat().st_mtime_ns, p.stat().st_size) for p in Path(OUT).rglob("window_*.json")}


snap_before = snapshot()
rc = main(argv)   # cùng argv, 10 file đã có
snap_after = snapshot()
print(f"files {len(snap_before)} -> {len(snap_after)} | rc={rc} (kỳ vọng: tập file không đổi, rc=0)")
assert rc == 0, f"resume lỗi rc={rc}"
assert set(snap_after) == set(snap_before), "resume ghi thêm/xoá file"
changed = [p for p in snap_before if snap_before[p] != snap_after[p]]
assert not changed, f"resume ghi đè {len(changed)} file: {changed[:3]}"
print("RESUME OK: 0 file bị ghi lại")


## A5b — full 307 cửa sổ

307 cửa sổ / 114 video; **65 cửa sổ dài > 30 s** (max 490 s) bị chia chunk theo D-1 → **480 chunk-call**.
Lượt chạy 31/10 trên T4: 198 cửa sổ trong lượt chính có median 3.2 s/cửa sổ (p90 6.9 s); tổng 3.101 s
gọi VLM cho cả 307 cửa sổ (~52 phút, chưa tính load model). Session chết → chạy lại cell sau, runner tự skip
file đã xong. Cell 11 in median/p90 trên toàn bộ file (gộp mọi lượt chạy, nên p90 cao hơn lượt chính).

- **Chạy bằng CLI** (không in-process) để log ra file thật, chứa dòng `xong: N window xét...`.
- Nếu quota fail: chạy `--shard 2/2` trên tài khoản Colab thứ hai (A3b); mỗi shard ghi file riêng nên ghép được.


In [ ]:
# Cell 10 — FULL RUN. Chạy lại cell này bao nhiêu lần tùy ý (resume).
# A5b cần #35: thiếu thì RoadAccidents002 + Vandalism017 lỗi và nghiệm thu ở cell 11 fail ở 305/307.
assert "clamp_window_to_video" in open("scripts/run_verify.py", encoding="utf-8").read(), (
    "main chưa merge #35 (cắt cửa sổ trôi quá cuối video) — không chạy A5b full. "
    "Nhờ Thịnh merge #35, `git pull --ff-only` rồi chạy lại cell này."
)
!python scripts/run_verify.py --manifest manifests/ucf_test.csv \
    --windows-dir results/windows --out-dir {OUT} --k 4 \
    --verifier-model Qwen/Qwen2.5-VL-3B-Instruct --continue-on-error 2>&1 | tee {LOG}


In [ ]:
# Cell 11 — NGHIỆM THU A5b: đủ 307 cửa sổ đúng danh sách, chi phí, tỷ lệ xác nhận.
import json
import math
import statistics
from pathlib import Path

files = sorted(Path(OUT).rglob("window_*.json"))
want = {(w["video_id"], w["window_index"])
        for w in json.load(open("results/windows/_index.json", encoding="utf-8-sig"))["windows"]}

seen, elapsed, confirmed, chunked, chunk_calls, configs, fakes = set(), [], 0, 0, 0, set(), 0
for f in files:
    d = json.loads(f.read_text(encoding="utf-8-sig"))
    meta = d["metadata"]   # layout A3b đã chốt — thiếu key là lỗi, không fallback
    seen.add((d["video_id"], meta["window_index"]))
    elapsed.append(meta["elapsed_s"])
    chunked += int(meta["n_chunks"] > 1)
    chunk_calls += meta["n_chunks"]
    confirmed += int(any(v["is_anomaly"] is True for v in d["verify_results"]))
    configs.add(json.dumps([meta["model"], meta["k"], meta["verifier_config"], meta["max_chunk_s"]], sort_keys=True))
    fakes += int(meta.get("fake") is True)

print(f"verify files: {len(files)} / {len(want)} cửa sổ | cấu hình khác nhau: {len(configs)} | fake: {fakes}")
if elapsed:
    ordered = sorted(elapsed)
    print("s/cửa sổ (metadata.elapsed_s): median", round(statistics.median(ordered), 2),
          "| p90", round(ordered[min(len(ordered) - 1, math.ceil(0.9 * len(ordered)) - 1)], 2))
print(f"cửa sổ bị chia chunk: {chunked}/{len(files)} = {100 * chunked / max(1, len(files)):.1f}% | chunk-call: {chunk_calls}")
print(f"VLM xác nhận: {confirmed}/{len(files)} = {100 * confirmed / max(1, len(files)):.1f}%")

assert fakes == 0, "có file --fake trong out-dir — không được nghiệm thu"
assert len(configs) == 1, "trộn nhiều cấu hình verifier — kết quả không cùng fingerprint"
assert len(files) == len(want), f"THIẾU {len(want) - len(files)} cửa sổ — A5b CHƯA xong"
assert seen == want, f"lệch danh sách: thiếu {len(want - seen)}, thừa {len(seen - want)}"
print("NGHIỆM THU OK: đủ và đúng 307 cửa sổ")


In [ ]:
# Cell 12 — đưa kết quả về máy (zip -> tải về, hoặc commit lên nhánh nếu < 25 MB).
import shutil

shutil.make_archive("/content/verify_a5b", "zip", OUT)
print("-> /content/verify_a5b.zip")
from google.colab import files as colab_files

colab_files.download("/content/verify_a5b.zip")


## Bẫy đã biết

- **`confidence` của Qwen 3B gần như hằng số ~0.95** → không được present như tín hiệu (`min_confidence` 0.5 trong `policy.yaml` là vô hiệu).
- **`event_type` từ VLM bias nặng về Robbery** (0/8 đúng ở sanity) → alert lấy type từ **typer C3** (quyết định D-2); VLM chỉ quyết `is_anomaly` + caption.
- Parse hỏng 2 lần → `VLMVerifier` **raise** (không bịa kết quả); `--continue-on-error` để 1 cửa sổ lỗi không kill cả batch, exit code 2 = có lỗi.
- Không bao giờ ghi output vào `results/verify/` bằng `--fake` (script tự chặn); kết quả thật phải về `results/verify/`.
- Cửa sổ dài: 65/307 > 30 s (max 490 s) → A3b chia chunk ≤ 30 s, verdict = OR; `metadata["n_chunks"]`/`elapsed_s` là nguồn đo duy nhất.
- Cửa sổ trôi quá cuối video (`RoadAccidents002`, `Vandalism017`): cần #35 cho A5b; cell 2 chỉ cảnh báo, cell full run (cell 10) assert và dừng ngay nếu thiếu.
- **Resume theo fingerprint**: đổi model / prompt / `k` / `max_chunk_s` → chạy lại từ đầu, không trộn kết quả cũ.

## Sau khi A5b xong

1. Tải `verify/` vào repo → `results/verify/`.
2. **A6a**: rút mẫu 20 cửa sổ (10 confirmed / 10 rejected, seed cố định) vào `notes/a6-handcheck-sample.csv` TRƯỚC khi đọc caption.
3. Báo Thịnh chạy `build_results.py --verify-dir results/verify` (C7d) → dashboard có caption/alert thật.
